# 02A Data Quality Audit

This notebook isolates the data-quality audit before transformation so the team can clearly show what was wrong in the raw Olist data and why each cleaning decision was needed.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv')
reviews = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv')
order_items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')


In [2]:
# Table-level snapshot.
audit_df = pd.DataFrame([
    {'table': 'orders', 'rows': len(orders), 'columns': len(orders.columns), 'duplicates': int(orders.duplicated().sum()), 'missing_cells': int(orders.isna().sum().sum())},
    {'table': 'reviews', 'rows': len(reviews), 'columns': len(reviews.columns), 'duplicates': int(reviews.duplicated().sum()), 'missing_cells': int(reviews.isna().sum().sum())},
    {'table': 'products', 'rows': len(products), 'columns': len(products.columns), 'duplicates': int(products.duplicated().sum()), 'missing_cells': int(products.isna().sum().sum())},
    {'table': 'order_items', 'rows': len(order_items), 'columns': len(order_items.columns), 'duplicates': int(order_items.duplicated().sum()), 'missing_cells': int(order_items.isna().sum().sum())},
    {'table': 'payments', 'rows': len(payments), 'columns': len(payments.columns), 'duplicates': int(payments.duplicated().sum()), 'missing_cells': int(payments.isna().sum().sum())},
])
audit_df


,table,rows,columns,duplicates,missing_cells
0,orders,99441,8,0,4908
1,reviews,99224,7,0,145903
2,products,32951,9,0,2448
3,order_items,112650,7,0,0
4,payments,103886,5,0,0


In [3]:
# Column-level missing value audit.
def missing_summary(df, name):
    out = df.isna().sum().reset_index()
    out.columns = ['column_name', 'missing_count']
    out['missing_pct'] = (out['missing_count'] / len(df) * 100).round(2)
    out['table'] = name
    return out[out['missing_count'] > 0].sort_values('missing_count', ascending=False)

missing_report = pd.concat([
    missing_summary(orders, 'orders'),
    missing_summary(reviews, 'reviews'),
    missing_summary(products, 'products')
], ignore_index=True)
missing_report


,column_name,missing_count,missing_pct,table
0,order_delivered_customer_date,2965,2.98,orders
1,order_delivered_carrier_date,1783,1.79,orders
2,order_approved_at,160,0.16,orders
3,review_comment_title,87656,88.34,reviews
4,review_comment_message,58247,58.70,reviews
5,product_category_name,610,1.85,products
6,product_name_lenght,610,1.85,products
7,product_description_lenght,610,1.85,products
8,product_photos_qty,610,1.85,products
9,product_weight_g,2,0.01,products


In [4]:
# Business-logic validation: orders missing delivery timestamps are mostly undelivered statuses.
orders['order_status'].value_counts()


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [5]:
orders.loc[orders['order_delivered_customer_date'].isna(), 'order_status'].value_counts()


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [6]:
# One-to-many join risk checks.
join_risk = {
    'orders_unique_order_id': int(orders['order_id'].nunique()),
    'order_items_unique_order_id': int(order_items['order_id'].nunique()),
    'payments_unique_order_id': int(payments['order_id'].nunique()),
    'reviews_unique_order_id': int(reviews['order_id'].nunique()),
    'max_items_per_order': int(order_items.groupby('order_id').size().max()),
    'max_payments_per_order': int(payments.groupby('order_id').size().max()),
}
join_risk


{'orders_unique_order_id': 99441,
 'order_items_unique_order_id': 98666,
 'payments_unique_order_id': 99440,
 'reviews_unique_order_id': 98673,
 'max_items_per_order': 21,
 'max_payments_per_order': 29}

## Audit Conclusions

- `orders` contains meaningful nulls tied to incomplete order lifecycles.
- `reviews` has many blank text comments, but that is behaviorally meaningful rather than random corruption.
- `products` has partial metadata gaps that should be flagged and imputed carefully.
- `order_items`, `payments`, and `reviews` all introduce one-to-many relationships, so aggregation to order level is required before Tableau use.
